# Repeated-cut assignment-aware prediction

## Goal checkpoint

Repeated sections are partial observations of a shared ordered physical PT object, with unknown nephron identities integrated into its distribution. This experiment tests the molecular prediction implied by ambiguous section assignments, not a new curve or a joint final estimator. Source-only training and transfer to the second healthy control remain a validation special case.

Notebooks 22/23 decode at the mean assigned position. For a nonlinear response curve, this differs from averaging decoded responses over possible positions. Keep each trained axis, projector, assignment and response decoder fixed, and compare those two reductions. Apply the same change to DPT/scFates neighbors with their actual inverse-distance assignments. Neighbor weights and conditional Gaussian/NB posteriors are uncalibrated model summaries. Better prediction cannot establish better spatial ordering.


## Frozen protocol and limits

Use actual healthy controls only, both transfer directions and the same three gene folds, excluding canonical markers from modeling. Source-only count PCs, scFates/DPT fits, bandwidth 0.08, NB dispersion 100, minimum support 12, 15 neighbors and source-only response splines stay fixed. Reproduce original point scores and positions, paired quarter-read scores, and the frozen selected-gene programs before interpreting averaging.

Predict **log-normalized responses** at the mean position or average the same response decoder over assignments. The two methods therefore share exact coordinates and ordering. No new parameter, query prior, specimen calibration, coverage matching or model selection is introduced. Segment and quadratic coverage comparators explicitly use query labels; molecular inference does not. Quarter-read predictions use original response measurements as the targets and freeze every source fit.

All-gene exposure still uses held-out counts; coarse source anatomy inherited molecular information. Panels were selected with the earlier atlas and reused for development. Gene folds are not independent mice. Uniform grid mixture remains a likelihood assumption, not a physical sampling law. Canonical spacing is not nephron length, and assignment entropy is not calibrated anatomical uncertainty. No depth, disease, human or TF branch follows from this response-reduction check.


In [ ]:
from pathlib import Path
import hashlib, io, json, sys
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs
from pseudospace.count_representation import fit_count_representation,transform_count_representation
from pseudospace.repeated_structure import fit_atlas
from pseudospace.trajectory_benchmark import fit_scfates_axis,fit_dpt_axis,project_neighbor_axis
from pseudospace.canonical_mean_atlas import fit_mean_atlas,project_mean_atlas
from pseudospace.count_rate_atlas import fit_count_rate_atlas,project_count_rate_atlas
from pseudospace.stage_cache import cached_payload
from importlib.metadata import version
import argparse,os
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root',default=os.environ.get('PSEUDOSPACE_DATA_ROOT',str(REPO/'data')))
parser.add_argument('--results-root',default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'repeated_structure_assignment_prediction'
OUT.mkdir(parents=True,exist_ok=True)
REFERENCE_OUT=RESULTS_ROOT/'healthy_reconstruction_trajectory_benchmark'
NOTEBOOK_LOGIC_VERSION='27.assignment_prediction.1'
SEED,N_COMPONENTS,N_FOLDS=15,15,3
CONTROLS=('Ctrl1A2','Ctrl1A4')
GRID=np.linspace(0,1,101)
SCFATES_PARAMS=dict(nodes=30,seed=SEED)
COUNT_ATLAS_PARAMS=dict(grid=GRID,bandwidth=.08,theta=100.,min_effective=12.)
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
VERSIONS={name:version(name) for name in ['scanpy','scFates','anndata','scikit-learn']}
CODE_FILES=[REPO/'pseudospace'/f for f in ['repeated_inputs.py','count_representation.py','trajectory_benchmark.py','repeated_structure.py','canonical_mean_atlas.py','count_rate_atlas.py','stage_cache.py']]
CODE_DIGEST=hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
BASELINE_SCORE_PATH=REFERENCE_OUT/'scores.csv'
if not BASELINE_SCORE_PATH.is_file(): raise FileNotFoundError(f'Required notebook-21 baseline is missing: {BASELINE_SCORE_PATH}')
BASELINE_SCORE_DIGEST=hashlib.sha256(BASELINE_SCORE_PATH.read_bytes()).hexdigest()
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,controls=CONTROLS,seed=SEED,
    components=N_COMPONENTS,folds=N_FOLDS,markers=sorted(MARKERS),grid=GRID.tolist(),
    scfates=SCFATES_PARAMS,mean_atlas=dict(bandwidth=.08,min_effective=12.),
    count_atlas={**COUNT_ATLAS_PARAMS,'grid':GRID.tolist()},
    count_representation_theta=100.,projection_neighbors=15,gene_spline_ridge=.001,
    quarter_read_fraction=.25,quarter_read_seed_rule='seed+fold',
    baseline_score_digest=BASELINE_SCORE_DIGEST,versions=VERSIONS)
print('Installed benchmark versions:',VERSIONS)

from pseudospace.within_segment_validation import fit_segment_covariate_oracle,predict_segment_covariate_oracle
LEGACY_OUT=RESULTS_ROOT/'scfates_repeated_count_reference'
LEGACY_FILES={name:LEGACY_OUT/(name+'.csv') for name in ['scores','thin_scores','positions','stability','fixed_program_scores']}
if any(not f.is_file() for f in LEGACY_FILES.values()):
    raise FileNotFoundError('Completed notebook-22 outputs are required')
legacy={name:pd.read_csv(path) for name,path in LEGACY_FILES.items()}
legacy_digests={name:hashlib.sha256(path.read_bytes()).hexdigest() for name,path in LEGACY_FILES.items()}
DPT_POSITION_PATH=REFERENCE_OUT/'positions.csv'
reference_positions=pd.read_csv(DPT_POSITION_PATH)
reference_scores=pd.read_csv(BASELINE_SCORE_PATH)
PANEL_PATH=RESULTS_ROOT/'within_segment_repeated_structure_validation'/'selected_genes.csv'
frozen_panel=pd.read_csv(PANEL_PATH,usecols=['source_specimen','test_specimen','fold','segment','gene','training_cv_gain'])
if frozen_panel.duplicated(['source_specimen','test_specimen','fold','segment','gene']).any():
    raise ValueError('Duplicated frozen program gene')
CODE_DIGEST=hashlib.sha256(CODE_DIGEST.encode()+(REPO/'pseudospace/within_segment_validation.py').read_bytes()).hexdigest()
CACHE_PARAMS.update(legacy_digests=legacy_digests,
 dpt_position_digest=hashlib.sha256(DPT_POSITION_PATH.read_bytes()).hexdigest(),
 panel_digest=hashlib.sha256(PANEL_PATH.read_bytes()).hexdigest(),dpt_neighbors=30,
 response_reduction=['decoder_at_assignment_mean','mean_of_decoded_assignments'],
 oracle=dict(degree=2,ridge=.01),depth_used=False)


## Validated actual inputs

In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if set(np.unique(specimens))!=set(CONTROLS): raise ValueError('Expected exactly the two healthy control specimens')
if not np.isin(labels,['S1','S2','S3']).all(): raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[x] for x in labels],dtype=int)
genes=healthy.var_names.to_numpy(str)
X=healthy.layers['lognorm']; X=X.toarray().astype(np.float64,copy=False) if sparse.issparse(X) else np.asarray(X,dtype=np.float64)
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()
if not np.isfinite(X).all() or not np.isfinite(raw_total).all() or np.any(raw_total<=0): raise ValueError('Invalid expression or all-gene exposure')
if 'n_spots' not in obs: raise ValueError('Required n_spots metadata is missing')
n_spots=pd.to_numeric(obs.n_spots,errors='coerce').to_numpy(float)
if not np.isfinite(n_spots).all() or np.any(n_spots<0): raise ValueError('Invalid n_spots metadata')
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(notebook_logic_version=NOTEBOOK_LOGIC_VERSION,config=CACHE_PARAMS,code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),dpt_role="source-only baseline",atlas_backbone="source-only scFates",depth_used=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab(specimens,labels))
print('Structures:',len(ids),'genes:',len(genes),'outer split: whole specimen')

In [ ]:
def dense_panel(matrix,rows,columns):
    value=matrix[rows][:,columns]
    return value.toarray() if sparse.issparse(value) else np.asarray(value)

def fit_gene_spline(response,z,sample):
    return fit_atlas(response,z,sample,grid=GRID,ridge=.001)

def normalized_variance(response,sample):
    mean=np.mean([response[sample==s].mean(axis=0) for s in np.unique(sample)],axis=0)
    return np.maximum(np.mean([((response[sample==s]-mean)**2).mean(axis=0)
                               for s in np.unique(sample)],axis=0),1e-3)

def primary_rows(response,predictions,variance,query_anatomy,meta):
    out=[]
    for method,prediction in predictions.items():
        scaled=(response-prediction)**2/variance
        for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
            use=np.ones(len(response),bool) if seg is None else query_anatomy==seg
            if use.any():
                out.append(dict(**meta,method=method,segment=name,n=int(use.sum()),
                                normalized_mse=float(scaled[use].mean())))
    return out

def selected_rows(response,predictions,train_response,train_anatomy,eval_anatomy,panel,gene_to_j,meta):
    out=[]
    for seg,name in enumerate(['S1','S2','S3']):
        selected=panel[panel.segment==name]
        if selected.empty or not set(selected.gene).issubset(gene_to_j):
            raise ValueError('Invalid frozen panel for '+name)
        jj=np.array([gene_to_j[g] for g in selected.gene],int)
        variance=np.maximum(np.var(train_response[train_anatomy==seg],axis=0),1e-3)[jj]
        use=eval_anatomy==seg
        if not use.any(): raise ValueError('Missing evaluation segment '+name)
        for method,prediction in predictions.items():
            out.append(dict(**meta,method=method,segment=name+'_selected',n=int(use.sum()),
              selected_gene_count=len(jj),normalized_mse=float(np.mean(
               (response[use][:,jj]-prediction[use][:,jj])**2/variance))))
    return out


def assignment_predictions(assignments,decoders):
    out={}
    for method,a in assignments.items():
        decoder=decoders[method]
        out[method]=decoder.predict(a['z_mean'])
        out[method+'_assignment']=decoder.predict_distribution(a['support'],a['weights'])
    return out

def grid_assignment(projected):
    return dict(z_mean=projected['z_mean'],support=GRID,weights=projected['posterior'])

def check_scores(current,previous,key,methods):
    saved=previous[(previous.source_specimen==key['source_specimen']) &
        (previous.test_specimen==key['test_specimen']) & (previous.fold==key['fold']) &
        previous.method.isin(methods)]
    current=current[current.method.isin(methods)]
    match=current.merge(saved,on=['method','segment'],validate='one_to_one',suffixes=('_new','_old'))
    if len(match)!=4*len(methods) or not np.allclose(match.normalized_mse_new,match.normalized_mse_old,rtol=1e-8,atol=1e-10):
        raise ValueError('Frozen point-response scores do not reproduce')


## Frozen-source transfers and assignment-aware decoding

In [ ]:
def run_assignment_prediction():
    frames={k:[] for k in ['scores','thin_scores','selected_scores','thin_selected_scores','stability','positions','diagnostics','failures']}
    old_methods=['scFates_neighbor','mean_reference','count_NB_reference']
    for source in CONTROLS:
        target=next(s for s in CONTROLS if s!=source)
        tr,te=specimens==source,specimens==target
        for fold in range(N_FOLDS):
            key=dict(source_specimen=source,test_specimen=target,fold=fold)
            try:
                mg=np.flatnonzero(~marker_mask&(gene_fold!=fold));eg=np.flatnonzero(~marker_mask&(gene_fold==fold))
                C=dense_panel(raw_counts,tr,mg);Q=dense_panel(raw_counts,te,mg)
                Y,transform=fit_count_representation(C,raw_total[tr],specimens[tr],N_COMPONENTS,theta=100.,seed=SEED)
                qY=transform_count_representation(Q,raw_total[te],transform)
                response=X[tr][:,eg];qresponse=X[te][:,eg];variance=normalized_variance(response,specimens[tr])
                sz,sm=fit_scfates_axis(Y,anatomy[tr],**SCFATES_PARAMS)
                dz,dm=fit_dpt_axis(Y,anatomy[tr],n_neighbors=30,seed=SEED)
                sdec=fit_gene_spline(response,sz,specimens[tr]);ddec=fit_gene_spline(response,dz,specimens[tr])
                mean=fit_mean_atlas(Y,sz,grid=GRID,bandwidth=.08,min_effective=12)
                rate=fit_count_rate_atlas(C,raw_total[tr],sz,specimens[tr],**COUNT_ATLAS_PARAMS)
                assignments=dict(scFates_neighbor=project_neighbor_axis(Y,sz,qY,k=15,return_distribution=True),
                    DPT_neighbor=project_neighbor_axis(Y,dz,qY,k=15,return_distribution=True),
                    mean_reference=grid_assignment(project_mean_atlas(mean,qY)),
                    count_NB_reference=grid_assignment(project_count_rate_atlas(rate,Q,raw_total[te])))
                decoders={m:(ddec if m=='DPT_neighbor' else sdec) for m in assignments}
                pred=assignment_predictions(assignments,decoders)
                point_scores=pd.DataFrame(primary_rows(qresponse,pred,variance,anatomy[te],key))
                check_scores(point_scores,legacy['scores'],key,old_methods)
                check_scores(point_scores,reference_scores,key,['DPT_neighbor'])
                # Structure-ID alignment makes row order irrelevant to reproduction.
                for method,a in assignments.items():
                    previous=reference_positions if method=='DPT_neighbor' else legacy['positions']
                    saved=previous[(previous.source_specimen==source)&(previous.test_specimen==target)&
                        (previous.fold==fold)&(previous.method==method)].set_index('structure_id')
                    if saved.index.duplicated().any() or set(saved.index)!=set(ids[te]):
                        raise ValueError('Frozen positions do not align exactly')
                    if not np.allclose(a['z_mean'],saved.loc[ids[te],'z'],rtol=1e-8,atol=1e-10):
                        raise ValueError('Frozen positions do not reproduce')
                panel=frozen_panel[(frozen_panel.source_specimen==source)&(frozen_panel.test_specimen==target)&(frozen_panel.fold==fold)]
                gene_to_j={g:j for j,g in enumerate(genes[eg])}
                # Same privileged comparator; anatomy/coverage do not enter projection.
                train_cov=np.column_stack([np.log(raw_total[tr]),np.log1p(n_spots[tr]),np.log1p(np.asarray((raw_counts[tr][:,mg]>0).sum(axis=1)).ravel())])
                query_cov=np.column_stack([np.log(raw_total[te]),np.log1p(n_spots[te]),np.log1p(np.asarray((raw_counts[te][:,mg]>0).sum(axis=1)).ravel())])
                oracle=fit_segment_covariate_oracle(response,train_cov,anatomy[tr],specimens[tr],ridge=.01,degree=2)
                pred['segment_coverage_oracle']=predict_segment_covariate_oracle(oracle,query_cov,anatomy[te])
                segmeans={a:response[anatomy[tr]==a].mean(axis=0) for a in (0,1,2)}
                pred['segment_mean_oracle']=np.vstack([segmeans[a] for a in anatomy[te]])
                selected=pd.DataFrame(selected_rows(qresponse,pred,response,anatomy[tr],anatomy[te],panel,gene_to_j,key))
                saved=legacy['fixed_program_scores'].query('source_specimen==@source and test_specimen==@target and fold==@fold').copy()
                saved['segment']=saved.segment.str.replace('_selected','',regex=False)
                ours=selected.copy();ours['segment']=ours.segment.str.replace('_selected','',regex=False)
                # Program score columns match after standardizing segment labels.
                matched=ours[ours.method.isin(old_methods)].merge(saved[saved.method.isin(old_methods)],on=['method','segment'],validate='one_to_one',suffixes=('_new','_old'))
                if len(matched)!=9 or not np.allclose(matched.normalized_mse_new,matched.normalized_mse_old,rtol=1e-8,atol=1e-10):
                    raise ValueError('Frozen positional-program scores do not reproduce')
                full=sparse.csr_matrix(raw_counts[te]);rng=np.random.default_rng(SEED+fold)
                thin=sparse.csr_matrix((rng.binomial(full.data.astype(np.int64),.25),full.indices.copy(),full.indptr.copy()),shape=full.shape)
                thinlib=np.asarray(thin.sum(axis=1)).ravel().astype(float)
                if np.any(thinlib<=0):raise ValueError('Zero exposure after paired thinning')
                thinC=dense_panel(thin,np.ones(len(thinlib),bool),mg)
                thinY=transform_count_representation(thinC,thinlib,transform)
                thin_a=dict(scFates_neighbor=project_neighbor_axis(Y,sz,thinY,k=15,return_distribution=True),
                    DPT_neighbor=project_neighbor_axis(Y,dz,thinY,k=15,return_distribution=True),
                    mean_reference=grid_assignment(project_mean_atlas(mean,thinY)),
                    count_NB_reference=grid_assignment(project_count_rate_atlas(rate,thinC,thinlib)))
                thinpred=assignment_predictions(thin_a,decoders)
                thin_scores=pd.DataFrame(primary_rows(qresponse,thinpred,variance,anatomy[te],key))
                check_scores(thin_scores,legacy['thin_scores'],key,old_methods)
                for m in ['segment_coverage_oracle','segment_mean_oracle']:thinpred[m]=pred[m]
                # Only append after all frozen-fit guards pass.
                frames['scores'].extend(primary_rows(qresponse,pred,variance,anatomy[te],key))
                frames['thin_scores'].extend(primary_rows(qresponse,thinpred,variance,anatomy[te],key))
                frames['selected_scores'].extend(selected.to_dict('records'))
                frames['thin_selected_scores'].extend(selected_rows(qresponse,thinpred,response,anatomy[tr],anatomy[te],panel,gene_to_j,key))
                for method,a in assignments.items():
                    z=a['z_mean'];zt=thin_a[method]['z_mean'];w=a['weights'];support=a['support']
                    delta=support-z[:,None];v=(w*delta**2).sum(axis=1)
                    h=-np.sum(w*np.log(np.maximum(w,1e-300)),axis=1)
                    for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
                        use=np.ones(len(z),bool) if seg is None else anatomy[te]==seg
                        for label in [method,method+'_assignment']:
                            frames['stability'].append(dict(**key,method=label,segment=name,n=int(use.sum()),
                                coordinate_mae=float(np.abs(z[use]-zt[use]).mean()),
                                prediction_change_normalized_mse=float(np.mean((pred[label][use]-thinpred[label][use])**2/variance)),
                                mean_assignment_variance=float(v[use].mean())))
                    for sid,z0,z1,v0,h0,label in zip(ids[te],z,zt,v,h,labels[te]):
                        frames['positions'].append(dict(**key,method=method,structure_id=sid,segment=label,
                            z=float(z0),thinned_z=float(z1),assignment_variance=float(v0),assignment_entropy=float(h0)))
                frames['diagnostics'].append(dict(**key,status='complete',model_genes=len(mg),response_genes=len(eg),
                    training_n=int(tr.sum()),query_n=int(te.sum()),scfates_metadata=json.dumps(sm),dpt_metadata=json.dumps(dm)))
                print('Completed:',source,'to',target,'fold',fold,flush=True)
            except Exception as exc:
                frames['failures'].append(dict(**key,error=str(exc)))
                frames['diagnostics'].append(dict(**key,status='failed',error=str(exc)))
                print('Failed:',source,'to',target,'fold',fold,str(exc),flush=True)
    return {name:pd.DataFrame(rows).to_json(orient='table') for name,rows in frames.items()}


In [ ]:
payload=cached_payload('assignment_aware_prediction',run_assignment_prediction,
 root=OUT/'stage_cache',params=CACHE_PARAMS,
 inputs=dict(X=X,counts=raw_counts,library=raw_total,ids=ids,specimens=specimens,anatomy=anatomy,
  markers=marker_mask,gene_fold=gene_fold,n_spots=n_spots,panel=frozen_panel,
  legacy=legacy,reference_scores=reference_scores,reference_positions=reference_positions),code=CODE_DIGEST)
outputs={name:pd.read_json(io.StringIO(str(value)),orient='table') for name,value in payload.items()}
for name,frame in outputs.items():frame.to_csv(OUT/(name+'.csv'),index=False)
display(outputs['diagnostics']);display(outputs['failures'])
if len(outputs['diagnostics'])!=6 or not (outputs['diagnostics'].status=='complete').all():
    raise RuntimeError('Incomplete frozen-fit reproduction; inspect saved failures before interpretation')
# Uncached reproduction guards also audit loaded stage-cache results.
audit_keys=['source_specimen','test_specimen','fold','method','segment']
old_methods=['scFates_neighbor','mean_reference','count_NB_reference']
for name,methods,previous,expected in [
    ('scores',old_methods,legacy['scores'],72),
    ('thin_scores',old_methods,legacy['thin_scores'],72),
    ('scores',['DPT_neighbor'],reference_scores,24)]:
    matched=outputs[name][outputs[name].method.isin(methods)].merge(
        previous[previous.method.isin(methods)],on=audit_keys,validate='one_to_one',suffixes=('_new','_old'))
    if len(matched)!=expected or not np.allclose(matched.normalized_mse_new,matched.normalized_mse_old,rtol=1e-8,atol=1e-10):
        raise ValueError('Uncached score reproduction failed: '+name)
saved_program=legacy['fixed_program_scores'].copy()
saved_program['method']=saved_program.method.replace({'quadratic_segment_coverage_oracle':'segment_coverage_oracle'})
matched=outputs['selected_scores'].merge(saved_program,on=audit_keys,validate='one_to_one',suffixes=('_new','_old'))
if len(matched)!=72 or not np.allclose(matched.normalized_mse_new,matched.normalized_mse_old,rtol=1e-8,atol=1e-10):
    raise ValueError('Uncached program/coverage reproduction failed')
position_keys=['source_specimen','test_specimen','fold','method','structure_id']
for previous,methods in [(legacy['positions'],old_methods),(reference_positions,['DPT_neighbor'])]:
    expected=previous[previous.method.isin(methods)]
    actual=outputs['positions'][outputs['positions'].method.isin(methods)]
    matched=actual.merge(expected,on=position_keys,validate='one_to_one',suffixes=('_new','_old'))
    if len(matched)!=len(expected) or len(matched)!=len(actual) or not np.allclose(matched.z_new,matched.z_old,rtol=1e-8,atol=1e-10):
        raise ValueError('Uncached structure-ID/position reproduction failed')
matched=outputs['stability'][outputs['stability'].method.isin(old_methods)].merge(
    legacy['stability'][legacy['stability'].method.isin(old_methods)],on=audit_keys,validate='one_to_one',suffixes=('_new','_old'))
if len(matched)!=72 or not np.allclose(matched.coordinate_mae_new,matched.coordinate_mae_old,rtol=1e-8,atol=1e-10):
    raise ValueError('Uncached paired-read movement reproduction failed')
print('Completed: uncached frozen-score, coordinate, program, coverage and read checks')
method_pairs=[]
for m in ['scFates_neighbor','DPT_neighbor','mean_reference','count_NB_reference']:
    method_pairs.append((m+'_assignment',m))
for candidate in ['mean_reference_assignment','count_NB_reference_assignment','scFates_neighbor_assignment']:
    for baseline in ['scFates_neighbor_assignment','DPT_neighbor_assignment','segment_mean_oracle','segment_coverage_oracle']:
        if candidate!=baseline: method_pairs.append((candidate,baseline))
keys=['source_specimen','test_specimen','fold','segment']
for name in ['scores','thin_scores','selected_scores','thin_selected_scores']:
    frame=outputs[name];parts=[]
    for candidate,baseline in method_pairs:
        left=frame[frame.method==candidate][keys+['normalized_mse']]
        right=frame[frame.method==baseline][keys+['normalized_mse']]
        match=left.merge(right,on=keys,validate='one_to_one',suffixes=('_candidate','_baseline'))
        match['candidate']=candidate;match['baseline']=baseline
        match['gain_percent']=100*(1-match.normalized_mse_candidate/match.normalized_mse_baseline)
        parts.append(match)
    matched=pd.concat(parts,ignore_index=True)
    matched.to_csv(OUT/(name+'_matched.csv'),index=False)
    summary=matched.groupby(['source_specimen','segment','candidate','baseline']).agg(
        gain_percent=('gain_percent','mean'),fold_count=('fold','nunique')).reset_index()
    summary.to_csv(OUT/(name+'_summary.csv'),index=False)
    display(summary)


## Focused program comparison

In [ ]:
import matplotlib.pyplot as plt
summary=pd.read_csv(OUT/'selected_scores_summary.csv')
fig,axes=plt.subplots(1,2,figsize=(12,4),sharey=True)
for ax,source in zip(axes,CONTROLS):
    sub=summary[(summary.source_specimen==source)&(summary.baseline=='scFates_neighbor_assignment')]
    for method in ['mean_reference_assignment','count_NB_reference_assignment']:
        rows=sub[sub.candidate==method].set_index('segment').reindex(['S1_selected','S2_selected','S3_selected'])
        ax.plot(['S1','S2','S3'],rows.gain_percent,marker='o',label=('Mean reference' if method=='mean_reference_assignment' else 'NB count reference'))
    ax.axhline(0,color='gray',lw=1);ax.set_title('Source '+source)
    ax.set_ylabel('Program error reduction (%)')
axes[-1].legend(frameon=False)
fig.suptitle('Assignment-averaged predictions: gains versus scFates')
fig.tight_layout();fig.savefig(OUT/'assignment_program_gains.png',dpi=180);plt.show()


## Measured result and decision

All six actual-control transfers pass the frozen-coordinate, point-score, selected-program and paired quarter-read reproduction checks. No positions or spatial ordering change.

Assignment averaging improves scFates all-gene errors by 0.143%/0.210% in the two source directions. Its selected-program gains relative to prediction at the mean coordinate are +0.34%/+7.50%/+3.20% (S1/S2/S3, Ctrl1A2 source) and +4.52%/+4.25%/+3.42% (reverse), with three folds each. DPT also improves aggregate prediction, but selected-program gains are smaller and heterogeneous.

This does not rescue Gaussian/NB projection. Against assignment-aware scFates, mean-reference all-gene changes are −0.116%/−0.157%, and NB changes −0.388%/−0.645%. For selected programs the NB changes are −1.75%/−5.97%/+0.91% and −5.94%/−25.74%/−2.72%; mean-reference changes are −1.05%/−6.37%/−1.89% and −1.19%/−13.57%/−2.52%. Some thinned-query S1/S3 comparisons favor NB, while the reverse S2 failure persists. Read robustness remains distinct from accurate program prediction.

Use assignment-aware prediction as an explicit response-reduction option in further comparisons; keep historical point scores and names intact. This improves evaluation of ambiguous repeated cuts, not reconstruction or calibration of their physical positions. The probabilistic projectors and unconstrained joint Gaussian hierarchy remain unadopted. The next experiment must address how the canonical ordered curve is constructed from repeated observations, rather than adding another response projector. The ledger specifies the next hypothesis before any new fits.
